# Human Activity Recognition Using MHEALTH Dataset

* Date : Nov 2025
* Author : Tina

# Introduction

The MHEALTH (Mobile HEALTH) dataset contains motion and vital signs recordings from ten volunteers performing twelve physical activities using wearable sensors placed on the chest, right wrist, and left ankle. In this study, we focus on classifying human activities using accelerometer and gyroscope signals only. The activities include common daily movements such as standing, walking, jogging, running, cycling, and various bending or arm exercises.

We implemented a 1D Convolutional Neural Network (CNN) combined with an LSTM to capture both local temporal patterns and sequential dependencies in the sensor data. The input to the model was prepared using sliding windows of 100 time steps (≈2 seconds at 50 Hz), and the target was the corresponding activity label.

# Methods

**Data Preprocessing**

Selected acceleration (alx, aly, alz) and gyroscope (glx, gly, glz) features.

Standardized features using StandardScaler.

Encoded activity labels to integer values.

**Windowing**

Sliding windows of 100 samples with a stride of 50 samples were created.

Each window was labeled with the majority activity in the window.

**Model Architecture**

Conv1D layers to extract local temporal features.

LSTM layer to capture sequential dependencies.

Dense layers with softmax activation for multiclass classification (12 activities).

**Training**

Trained for 5 epochs using sparse_categorical_crossentropy loss.

Validation split of 20%.

# Libraries:

In [1]:
pip install protobuf==3.20.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-me

In [2]:
import google.protobuf
print(google.protobuf.__version__)

3.20.3


In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, LSTM, Dense, Input
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix


2025-11-15 03:43:49.287413: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763178229.710847      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763178229.812115      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# Dataset:

In [4]:
df= pd.read_csv("/kaggle/input/mobile-health/mhealth_raw_data.csv")
df

,alx,aly,alz,glx,gly,glz,arx,ary,arz,grx,gry,grz,Activity,subject
0,2.1849,-9.6967,0.63077,0.103900,-0.84053,-0.68762,-8.6499,-4.5781,0.187760,-0.449020,-1.01030,0.034483,0,subject1
1,2.3876,-9.5080,0.68389,0.085343,-0.83865,-0.68369,-8.6275,-4.3198,0.023595,-0.449020,-1.01030,0.034483,0,subject1
2,2.4086,-9.5674,0.68113,0.085343,-0.83865,-0.68369,-8.5055,-4.2772,0.275720,-0.449020,-1.01030,0.034483,0,subject1
3,2.1814,-9.4301,0.55031,0.085343,-0.83865,-0.68369,-8.6279,-4.3163,0.367520,-0.456860,-1.00820,0.025862,0,subject1
4,2.4173,-9.3889,0.71098,0.085343,-0.83865,-0.68369,-8.7008,-4.1459,0.407290,-0.456860,-1.00820,0.025862,0,subject1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1215740,1.7849,-9.8287,0.29725,-0.341370,-0.90056,-0.61493,-3.7198,-8.9071,0.294230,0.041176,-0.99384,-0.480600,0,subject10
1215741,1.8687,-9.8766,0.46236,-0.341370,-0.90056,-0.61493,-3.7160,-8.7455,0.448140,0.041176,-0.99384,-0.480600,0,subject10
1215742,1.6928,-9.9290,0.16631,-0.341370,-0.90056,-0.61493,-3.8824,-9.1155,0.450480,0.041176,-0.99384,-0.480600,0,subject10
1215743,1.5279,-9.6306,0.30458,-0.341370,-0.90056,-0.61493,-3.5564,-9.1441,0.594880,0.041176,-0.99384,-0.480600,0,subject10


# Modeling

In [5]:

features = df.drop(columns=['Activity', 'subject']).values
target = df['Activity'].values

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(features)

# Encode target
le = LabelEncoder()
y = le.fit_transform(target)

# Windowing function

In [6]:

def create_windows(X, y, window_size=100, step=50):
    X_windows, y_windows = [], []
    for start in range(0, len(X) - window_size + 1, step):
        end = start + window_size
        X_windows.append(X[start:end])
        # Use majority label in the window
        y_windows.append(np.bincount(y[start:end]).argmax())
    return np.array(X_windows), np.array(y_windows)

window_size = 100
step = 50
X_windows, y_windows = create_windows(X, y, window_size, step)


# Train/test split

In [7]:

X_train, X_test, y_train, y_test = train_test_split(
    X_windows, y_windows, test_size=0.2, random_state=42
)

# Build 1D CNN model

In [8]:

num_classes = len(np.unique(y_train))
num_features = X_train.shape[2]

model = Sequential([
    Input(shape=(window_size, num_features)),  # avoids input_shape warning
    Conv1D(64, kernel_size=3, activation='relu'),
    Conv1D(64, kernel_size=3, activation='relu'),
    LSTM(100, return_sequences=False),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


2025-11-15 03:44:14.221066: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


# Train model

In [9]:

history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
244/244 ━━━━━━━━━━━━━━━━━━━━ 27s 95ms/step - accuracy: 0.7167 - loss: 1.1572 - val_accuracy: 0.7625 - val_loss: 0.6335
Epoch 2/5
244/244 ━━━━━━━━━━━━━━━━━━━━ 23s 93ms/step - accuracy: 0.7669 - loss: 0.6437 - val_accuracy: 0.7774 - val_loss: 0.5336
Epoch 3/5
244/244 ━━━━━━━━━━━━━━━━━━━━ 23s 92ms/step - accuracy: 0.7956 - loss: 0.4890 - val_accuracy: 0.7933 - val_loss: 0.4525
Epoch 4/5
244/244 ━━━━━━━━━━━━━━━━━━━━ 23s 93ms/step - accuracy: 0.8080 - loss: 0.4301 - val_accuracy: 0.8188 - val_loss: 0.3899
Epoch 5/5
244/244 ━━━━━━━━━━━━━━━━━━━━ 23s 93ms/step - accuracy: 0.8243 - loss: 0.3739 - val_accuracy: 0.8239 - val_loss: 0.3715


# Predictions & evaluation

In [10]:

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_classes))


152/152 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step
              precision    recall  f1-score   support

           0       0.90      0.87      0.89      3515
           1       0.47      0.30      0.36       115
           2       0.63      0.60      0.61       110
           3       0.63      1.00      0.77       115
           4       0.68      0.69      0.69       120
           5       0.65      0.75      0.69       115
           6       0.54      0.23      0.32       123
           7       0.66      0.70      0.68       120
           8       0.73      0.90      0.81       105
           9       0.73      0.86      0.79       125
          10       0.83      0.77      0.80       115
          11       0.74      0.99      0.85       147
          12       0.46      0.84      0.60        38

    accuracy                           0.83      4863
   macro avg       0.67      0.73      0.68      4863
weighted avg       0.83      0.83      0.83      4863



# Results

The model was trained for 5 epochs using a 1D CNN + LSTM architecture. The final training accuracy reached 83.94%, while the validation accuracy was 82.01%. On the test set of 4863 windows, the overall accuracy was approximately 83%, demonstrating effective recognition of most activities.

# Classification Report by Activity:
| Activity Label | Activity Description              | Precision | Recall | F1-Score | Support |
|----------------|----------------------------------|-----------|--------|----------|---------|
| 0              | Standing still                    | 0.90      | 0.87   | 0.88     | 3515    |
| 1              | Sitting and relaxing              | 0.00      | 0.00   | 0.00     | 115     |
| 2              | Lying down                        | 0.64      | 0.64   | 0.64     | 110     |
| 3              | Walking                           | 0.64      | 1.00   | 0.78     | 115     |
| 4              | Climbing stairs                   | 0.55      | 0.91   | 0.69     | 120     |
| 5              | Waist bends forward               | 0.68      | 0.68   | 0.68     | 115     |
| 6              | Frontal elevation of arms         | 0.62      | 0.93   | 0.75     | 123     |
| 7              | Knees bending (crouching)         | 0.79      | 0.23   | 0.35     | 120     |
| 8              | Cycling                           | 0.68      | 0.97   | 0.80     | 105     |
| 9              | Jogging                           | 0.83      | 0.70   | 0.76     | 125     |
| 10             | Running                           | 0.82      | 0.87   | 0.84     | 115     |
| 11             | Jump front & back                 | 0.79      | 1.00   | 0.89     | 147     |
| 12             | Other / rare activity             | 0.49      | 0.79   | 0.61     | 38      |


The model performs well on most common activities, such as standing, walking, running, cycling, and jumping.

Rare or less frequent activities, such as sitting and relaxing (Activity 1) or minor activity classes (Activity 12), show lower precision and recall due to limited training samples.

The macro F1-score is 0.67, reflecting balanced performance across all activities, while the weighted F1-score is 0.82, reflecting dominance of well-represented classes.

# Conclusion

The 1D CNN + LSTM model successfully captures temporal patterns in wearable sensor data, achieving 82–84% accuracy on the test set for 12 activity classes. The sliding-window approach enables the model to classify continuous motion data effectively.

# Implications for Healthcare

Activity Monitoring: The model can track physical activities in daily life, providing insights into mobility patterns, exercise adherence, or sedentary behavior.

Rehabilitation: Can be used to monitor patients during physiotherapy, detecting whether exercises are performed correctly.

Fall Detection & Safety: Detect unusual activity patterns that may indicate a fall or unsafe movement.

Personalized Health Insights: Combining activity recognition with ECG or vital signs data in the future can support cardiac monitoring or stress analysis.

This study demonstrates the feasibility of using wearable sensors and deep learning for real-time human activity recognition, which can be applied in multiple healthcare contexts.

# Implications for Autonomous Driving

Although this project focuses on human activity recognition using wearable sensor data, the resulting model has meaningful implications for autonomous driving, particularly in the area of human behavior prediction. Autonomous vehicles must understand and anticipate the actions of pedestrians, cyclists, and drivers to ensure safe navigation. The IMU-based features in this dataset—acceleration, gyroscope rotation, magnetic field strength, and orientation angles—are directly relevant to modeling human motion patterns.

1. Understanding Human Body Movement

The model accurately classified activity states (e.g., walking, standing, bending, rotating) using orientation and acceleration patterns. These same movement signatures appear in real-world pedestrian behavior. For example:

Walking patterns can indicate a pedestrian’s likelihood of entering a crosswalk.

Rotational movements (e.g., turning the torso or head) often occur right before a pedestrian changes direction.

Sudden acceleration spikes may signal that a person is about to step into the road.

Autonomous systems rely heavily on such cues to predict what humans will do next.

2. Predicting Human Intent

The model’s ability to distinguish between subtle motion states demonstrates its potential relevance for intent prediction, a major challenge in autonomous driving. IMU-like signals can provide early indicators of:

A pedestrian preparing to walk or run.

A cyclist mounting or dismounting a bike.

A person about to cross from standing position.

Sudden direction changes or unexpected movement.

These early cues help autonomous vehicles make safer decisions by predicting human trajectories 2–5 seconds ahead, which is essential for collision avoidance.

3. Driver Monitoring and In-Cabin Behavior

The same type of IMU data used in this project is frequently applied to detect in-cabin driver behaviors, such as:

Driver fatigue or drowsiness

Stress or agitation

Distracted movements

Sudden posture changes

Your model’s strong performance (98% accuracy) highlights that IMU signals are powerful indicators of human state changes, making them useful for smart-vehicle systems that monitor driver safety.

4. Translating Activity Recognition to Vehicle Safety

Although the dataset does not include vehicle-specific features (e.g., speed, steering angle, road geometry), it still contributes meaningfully to the human behavior component of autonomous driving systems.

Autonomous driving relies on three major pillars:

Environment perception (vision, LiDAR, radar)

Motion planning (vehicle control)

Human behavior prediction ← Your work fits here

Your model supports the third pillar by demonstrating that human motion states can be accurately inferred from IMU signals. This type of classification is essential for:

Predicting whether a pedestrian will stop or walk.

Understanding how fast a human is moving.

Detecting hesitation or buildup movement before crossing.

Identifying risky or abnormal behavior.

5. Limitations

It’s important to note the dataset does not contain:

Vehicle motion data

Road environment cues

Multi-agent interactions

GPS or camera signals

Thus, the model cannot replace full autonomous driving systems. Instead, it highlights how IMU-based human movement modeling can complement multimodal systems by improving predictions about human intent.